# 36 · Choosing the paper's PT coordinate, and what the choice changes downstream

Notebook 35 validated the current coordinate (SCF13, notebook 13's scFates curve). It recovers
external zonation as well as the segment labels do and absorbs no injected species effect, but it
fails the pre-registered equal-depth test (median within-segment agreement 0.72 < 0.80; human S1
ordering follows read depth), and its human-versus-mouse placement shifts by up to 0.29 when a
third of its genes or half its reads are removed. This notebook compares the alternatives fixed in
the workstream-1 plan and applies two rules that were written down before any of these outputs:

- **C6 (CAC adoption):** the conserved-anchor coordinate replaces SCF13 only if it is non-inferior,
  has an advantage not favoured by its construction, is stable, is robust to its own assumption and
  keeps a specific downstream screen. Notebook 35 already showed that no advantage criterion can be
  met, so CAC is reported as a robustness check unless it passes as written.
- **Paper-coordinate selection:** among SCF13-ED, DPT13 and (only if it passes C6) CAC, a candidate
  qualifies if P5 ≥ 0.80, P1 ≥ SCF13 − 0.05 in each species and its registration gap ≤ 0.15; the
  highest mean gene-fold within-segment agreement wins, preferring DPT over scFates within 0.02.
  If none qualifies, SCF13 stays and its limitations are reported.

The selected coordinate then goes through notebook 12's gene model and notebook 31's specificity
gate, and its retention of the 66 robust and 42 core pathways is reported. Registration and seed
stability are descriptive and were added after notebook 35's outputs.

**Cohort.** Two control mice and two cortex sections from one male human donor; specimens, not
structures, are replicates; everything is descriptive.

## 1 · Folders and fixed settings

Pass `--data-root` / `--results-root` or set `PSEUDOSPACE_DATA_ROOT` / `PSEUDOSPACE_RESULTS_ROOT`.

In [ ]:
import argparse
import json
import math
import os
import sys
from hashlib import sha256
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream12 = results_root / 'pt_pathway_remodeling_scfates'
upstream33 = results_root / 'pt_pathway_registration_sensitivity'
external = data_root / 'external'
output = results_root / 'pt_reconstruction_v2' / 'nb36'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
SAMPLES = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
source_paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                for names in SAMPLES.values() for n in names}
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'cross_species_nephron_scfates.h5ad',
            upstream13 / 'ortholog_map_used.csv', upstream12 / 'gene_statistics.csv',
            upstream33 / 'confident_pathways_with_robustness.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            results_root / 'pt_reconstruction_v2' / 'nb35' / 'p2_absorption.csv',
            results_root / 'pt_reconstruction_v2' / 'nb35' / 'p2_displacement.csv',
            *source_paths.values()]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '36.coordinate_selection.1'  # Bump when a cached calculation changes.
SEED = 0                     # Notebook 13's seed for PCA, Harmony and scFates.
SCFATES_NODES, TRAJECTORY_DIMS, PT_NEIGHBORS = 30, 5, 30
EARLY_MARKERS, LATE_MARKERS = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_FOLDS = 3
MIN_DETECTION = .10          # Per species, in our PT structures.
ANCHOR_LOOSE, ANCHOR_STRICT = .5, 1.0
INJECTION_SHARE_MOD = 10     # Keep HVGs with sha256 mod 10 == 0.
INJECTION_LOG_AMPLITUDE = math.log(2)  # f(s) = log 2 * (2s - 1): 0.5x to 2x along the PT
ABSORPTION_FLOOR = .02
EQUAL_DEPTH_QUANTILE = .20
CAC_PCS = 20
SEEDS = (1, 2, 3, 4, 5)
SPLIT_EPSILON, SPLIT_SEED = .5, 35
N_NULL, PATHWAY_SEED, ALPHA, BASIS_DF = 9999, 12, .05, 6
HUMAN, MOUSE, INK = '#D55E00', '#0072B2', '#333333'
print('Results folder:', output)

## 2 · Inputs: notebook 13's structures, its pass-2 cache and the original counts

The pass-2 nephron embedding is read from notebook 13's own stage cache, identified by its
recorded logic version, and checked against the saved structure list. Raw counts are rebuilt from
the original specimen matrices for every nephron structure; on the 10,076 genes notebook 13 used
they must equal its cached counts exactly. Every perturbation below acts on these counts once.

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rpy2.robjects as ro
import scanpy as sc
import scFates as scf
from IPython.display import display
from scipy import sparse
from scipy.stats import spearmanr

import pseudospace.anchor_coordinate as anchor_module
import pseudospace.coordinate_arms as arms_module
import pseudospace.coordinate_evaluation as evaluation_module
from pseudospace.coordinate_arms import (fit_dpt, fit_scfates_curve, harmony_trajectory, heldout_prediction,
                                         loso_positions, marker_tip_score, registration_gap, scanpy_lognorm,
                                         species_intersection_hvgs, within_segment_agreement)
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle
from pseudospace.anchor_coordinate import (bulk_segment_profiles, census_segment_profiles,
                                           conserved_anchor_mask, positive_control_masks,
                                           pt_specific_mask, segment_contrast)
from pseudospace.coordinate_evaluation import (absorption_ratio, binomial_thin_to_depth, concordance,
                                               fold_noise_variance, group_rank_correlations,
                                               interpolate_curves, log_normalize, poisson_count_split,
                                               reflect_unit, scale_counts, sha256_folds)
from pseudospace.harmony import run_harmony_rpy2
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_frame, cached_payload, digest

installed_harmony = str(ro.r('as.character(packageVersion("harmony"))')[0])
if installed_harmony != '2.0.5' or scf.__version__ != '1.2.5':
    raise RuntimeError(f'Need R harmony 2.0.5 and scFates 1.2.5; found {installed_harmony}, {scf.__version__}.')

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
saved_neph = ad.read_h5ad(upstream13 / 'cross_species_nephron_scfates.h5ad', backed='r').obs.copy()
key_files = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
             if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if len(key_files) != 1:
    raise FileNotFoundError(f'Expected one notebook-13 pass-2 cache, found {len(key_files)}; rerun notebook 13.')
neph = ad.read_h5ad(str(key_files[0])[:-len('.key.json')])
if not neph.obs_names.equals(saved_neph.index):
    raise ValueError('Notebook 13 pass-2 cache does not match its saved nephron structures.')
pt_mask = neph.obs.coarse_class.astype(str).eq('PT').to_numpy()
pt_rows = np.flatnonzero(pt_mask)
if not neph.obs_names[pt_rows].equals(saved_pt.index):
    raise ValueError('PT rows differ from notebook 13 saved PT structures.')

rebuilt = rebuild_pt_expression(
    saved_neph[['sample', 'comparison_species', 'shared_pseudospace']].astype(
        {'sample': str, 'comparison_species': str}),
    source_paths, pd.read_csv(upstream13 / 'ortholog_map_used.csv'), target_sum=1e4)
COUNTS = sparse.csr_matrix(rebuilt.layers['counts']).astype(np.int64)
genes_all = pd.Index(rebuilt.var_names)
measured_both = rebuilt.var.measured_in_both_inputs.to_numpy(bool)
cache_genes = pd.Index(neph.var_names)
cache_cols = genes_all.get_indexer(cache_genes)
if (cache_cols < 0).any():
    raise ValueError('Notebook 13 genes missing from the rebuilt panel.')
if abs(COUNTS[:, cache_cols] - sparse.csr_matrix(neph.layers['counts'])).max() != 0:
    raise ValueError('Rebuilt counts differ from notebook 13 cached counts.')
ORIGINAL_HVGS = cache_genes[neph.var.highly_variable_for_harmony.to_numpy(bool)]
# scanpy's PCA masks to var['highly_variable'], which here is pass 1's top-2000 list: notebook 13's
# PCA therefore used the intersection. Those are the genes that actually build the coordinate.
CONSTRUCTION_GENES = cache_genes[(neph.var.highly_variable_for_harmony & neph.var.highly_variable).to_numpy(bool)]
neph_obs = neph.obs.copy()
neph_var = neph.var[['exclude_from_harmony_hvg', 'highly_variable']].copy()
del neph

species_pt = saved_pt.comparison_species.astype(str).to_numpy()
specimen_pt = saved_pt['sample'].astype(str).to_numpy()
segment_pt = saved_pt.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
human_pt = species_pt == 'human'
SCF_SAVED = saved_pt.shared_pseudospace.to_numpy(float)
DPT_SAVED = saved_pt.dpt_pseudospace.to_numpy(float)
print(f'{len(neph_obs):,} nephron structures ({len(pt_rows):,} PT) · {len(genes_all):,} rebuilt genes · '
      f'{len(ORIGINAL_HVGS)} notebook-13 Harmony HVGs, {len(CONSTRUCTION_GENES)} used by its PCA')


def pt_lognorm(counts):
    """Notebook-12 normalization for PT rows: log1p over the measured-in-both total x 1e4."""
    block = counts[pt_rows]
    library = np.asarray(block[:, measured_both].sum(axis=1)).ravel().astype(float)
    if (library <= 0).any():
        raise ValueError('A PT structure has no counts in the shared panel.')
    return log_normalize(block, library), library


EVAL_ORIGINAL, LIBRARY_PT = pt_lognorm(COUNTS)
detection = pd.DataFrame({sp: np.asarray((COUNTS[pt_rows][species_pt == sp] > 0).mean(axis=0)).ravel()
                          for sp in SAMPLES}, index=genes_all)

## 3 · External segment data and the fixed gene sets

Anchors are **selected** on mouse microdissection, mouse snRNA from female donors and human snRNA
donor half A. Everything is **evaluated** on mouse snRNA from male donors, human snRNA donor half B
and rat microdissection. Anchors and notebook 13's six orientation markers never enter an
evaluation set.

| Set | Rule |
|---|---|
| A166-rule / A49-rule anchors | same S3−early sign in all selection datasets, |Δ| ≥ 0.5 / ≥ 1, expressed (≥ 1 log2) in each, measured in both species, detected in ≥ 10% of each species' PT, PT-specific (no non-PT tubular class > 2× the PT mean in either species) |
| E (evaluation) | non-anchor, non-marker, measured in both, detected ≥ 10% in the evaluated species, expressed ≥ 1 in the reference |
| D (positive controls) | opposite sign human vs mouse (|Δ| ≥ 0.5 both), or mouse-zonated (|Δ| ≥ 1) and human-flat (|Δ| < 0.25), evaluation datasets |
| H (conserved held-out) | A49-rule genes that are not among the 611 genes notebook 13's PCA used |

In [ ]:
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_micro = bulk_segment_profiles(mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol'),
                                    '_PT{seg}_').reindex(genes_all)
rat_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz', sep='\t')
rat_micro = bulk_segment_profiles(rat_bulk.drop_duplicates('Symbol').set_index('Symbol'), '{seg}_')
rat_micro.index = rat_micro.index.astype(str).str.upper()
rat_micro = rat_micro[~rat_micro.index.duplicated()].reindex(genes_all.str.upper())
rat_micro.index = genes_all


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(genes_all)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
HALF_A, HALF_B = human_donors[0::2], human_donors[1::2]
if HALF_A != ['21_015', 'HCA_51_ref', 'PRECISE-34'] or HALF_B != ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36']:
    raise ValueError(f'Human donor halves differ from the pre-registered split: {HALF_A} / {HALF_B}')


def micro_contrast(frame, late, early):
    return frame[late] - frame[early], frame[[late, early]].max(axis=1)


ext = {  # name -> (contrast, level)
    'select_mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
    'select_mouse_sn_female': segment_contrast(mouse_sn, sex='female'),
    'select_human_sn_halfA': segment_contrast(human_sn, donors=HALF_A),
    'mouse_sn_male': segment_contrast(mouse_sn, sex='male'),
    'mouse_sn_male_S2mS1': segment_contrast(mouse_sn, late='S2', early='S1', sex='male'),
    'mouse_sn_male_S3mS1': segment_contrast(mouse_sn, late='S3', early='S1', sex='male'),
    'mouse_sn_male_S3mS2': segment_contrast(mouse_sn, late='S3', early='S2', sex='male'),
    'mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
    'mouse_micro_S2mS1': micro_contrast(mouse_micro, 'S2', 'S1'),
    'mouse_micro_S3mS1': micro_contrast(mouse_micro, 'S3', 'S1'),
    'mouse_micro_S3mS2': micro_contrast(mouse_micro, 'S3', 'S2'),
    'rat_micro': micro_contrast(rat_micro, 'S3', 'early'),
    'human_sn_halfB': segment_contrast(human_sn, donors=HALF_B),
}
contrast = pd.DataFrame({k: v[0] for k, v in ext.items()})
level = pd.DataFrame({k: v[1] for k, v in ext.items()})

# Position-free PT-specificity: linear-scale class means per specimen, averaged within species.
classes = neph_obs.coarse_class.astype(str).to_numpy()
specimen_neph = neph_obs['sample'].astype(str).to_numpy()
library_neph = np.asarray(COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
normalized = sparse.csr_matrix(COUNTS.multiply((1e4 / library_neph)[:, None]))
class_means = {}
for sp, names in SAMPLES.items():
    columns = {}
    for cls in sorted(set(classes)):
        per = [np.asarray(normalized[(classes == cls) & (specimen_neph == n)].mean(axis=0)).ravel()
               for n in names if ((classes == cls) & (specimen_neph == n)).sum() >= 20]
        columns[cls] = np.mean(per, axis=0)
    class_means[sp] = pd.DataFrame(columns, index=genes_all)
del normalized
eligible_anchor = (pd.Series(measured_both, index=genes_all) & detection.mouse.ge(MIN_DETECTION)
                   & detection.human.ge(MIN_DETECTION) & pt_specific_mask(class_means))
selection = ['select_mouse_micro', 'select_mouse_sn_female', 'select_human_sn_halfA']
A_LOOSE = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_LOOSE)
A_STRICT = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_STRICT)
markers = pd.Series(genes_all.isin(EARLY_MARKERS + LATE_MARKERS), index=genes_all)
anchor_sign = np.sign(contrast['select_human_sn_halfA'])


def evaluation_mask(sp, reference):
    return (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection[sp].ge(MIN_DETECTION)
            & level[reference].ge(1) & contrast[reference].notna())


control_eligible = (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers
                    & detection.mouse.ge(MIN_DETECTION) & detection.human.ge(MIN_DETECTION)
                    & level['mouse_sn_male'].ge(1))
D_OPPOSITE, D_FLAT = positive_control_masks(contrast['human_sn_halfB'], contrast['mouse_sn_male'],
                                            contrast['mouse_micro'], level['human_sn_halfB'], control_eligible)
H_SET = A_STRICT & ~pd.Series(genes_all.isin(CONSTRUCTION_GENES), index=genes_all)
FOLD = pd.Series(sha256_folds(genes_all, N_FOLDS), index=genes_all)
gene_sets_table = pd.DataFrame({
    'anchor_loose_A166rule': A_LOOSE, 'anchor_strict_A49rule': A_STRICT, 'positive_opposite': D_OPPOSITE,
    'positive_flat': D_FLAT, 'conserved_heldout_H': H_SET, 'fold': FOLD,
    'in_notebook13_construction_genes': genes_all.isin(CONSTRUCTION_GENES), **contrast.add_prefix('contrast_'),
    **detection.add_prefix('detection_')})
gene_sets_table.to_csv(output / 'gene_sets_and_external_contrasts.csv')
set_counts = pd.Series({
    'anchors |Δ|≥0.5 (A166 rule)': int(A_LOOSE.sum()),
    'anchors |Δ|≥0.5, late-high': int((A_LOOSE & anchor_sign.gt(0)).sum()),
    'anchors |Δ|≥1 (A49 rule)': int(A_STRICT.sum()),
    'anchors |Δ|≥1, late-high': int((A_STRICT & anchor_sign.gt(0)).sum()),
    'notebook-13 construction genes (PCA)': len(CONSTRUCTION_GENES),
    'strict anchors among construction genes': int((A_STRICT & genes_all.isin(CONSTRUCTION_GENES)).sum()),
    'positive controls: opposite sign': int(D_OPPOSITE.sum()),
    'positive controls: mouse-zonated, human-flat': int(D_FLAT.sum()),
    'conserved held-out H': int(H_SET.sum()),
    'E mouse (male snRNA reference)': int(evaluation_mask('mouse', 'mouse_sn_male').sum()),
    'E human (snRNA half B reference)': int(evaluation_mask('human', 'human_sn_halfB').sum())}, name='genes')
set_counts.to_csv(output / 'gene_set_counts.csv')
display(set_counts.to_frame())

## 4 · The coordinate arms

| Arm | Genes and structures that build the curve | Root |
|---|---|---|
| SCF13 | notebook 13: species-intersection HVGs on the whole nephron, PCA on those that are also pass 1's top-2000 HVGs (611), Harmony, 5 dimensions, 30-node scFates | marker tip score |
| SCF13-ED | the same pipeline on counts thinned to equal depth (pooled PT 20th percentile) | marker tip score |
| DPT13 | DPT on SCF13's embedding, from SCF13's root structure | — |
| SCF-PT | notebook 13's pipeline on PT structures only (scope control) | marker tip score |
| CAC | PT only; the conserved anchors alone; PCA 20 (no mask), Harmony, 5 dimensions, scFates | tip with the higher early − late anchor score, which both species must pick |
| CAC-S49 / CAC-C | strict anchors / no Harmony, specimens centred on their S1 and S2 means | as CAC |

All arms read the same raw counts. Every refit is cached; guards run on every execution.

In [ ]:
cache_root = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(arms_module.__file__), Path(evaluation_module.__file__),
               Path(anchor_module.__file__), Path(run_harmony_rpy2.__code__.co_filename)])
BASE_DIGEST = digest(COUNTS)
PASS1_HV = neph_var.highly_variable.to_numpy(bool)
EXCLUDE_BASE = neph_var.exclude_from_harmony_hvg.to_numpy(bool)
marker_cols = cache_genes.get_indexer(list(EARLY_MARKERS + LATE_MARKERS))
ANCHORS = [g for g in genes_all[A_LOOSE.to_numpy()] if g in set(cache_genes)]
ANCHORS_STRICT = [g for g in genes_all[A_STRICT.to_numpy()] if g in set(cache_genes)]
print(f'{len(ANCHORS)} of {int(A_LOOSE.sum())} anchors are among notebook 13 genes; strict: {len(ANCHORS_STRICT)}')


def anchor_score(lognorm_pt, anchors):
    idx = cache_genes.get_indexer(anchors)
    sign = anchor_sign.loc[list(anchors)].to_numpy()
    early = np.asarray(lognorm_pt[:, idx[sign < 0]].mean(axis=1)).ravel()
    late = np.asarray(lognorm_pt[:, idx[sign > 0]].mean(axis=1)).ravel()
    return early - late


def embed(arm, counts, exclude=(), anchors=None):
    """Return the PT trajectory representation, tip score, root groups and construction genes."""
    lognorm = scanpy_lognorm(counts[:, cache_cols])
    excluded = EXCLUDE_BASE | cache_genes.isin(list(exclude))
    if arm in ('nephron', 'pt'):
        rows = np.arange(len(neph_obs)) if arm == 'nephron' else pt_rows
        block, obs = lognorm[rows], neph_obs.iloc[rows]
        hvg = species_intersection_hvgs(block, obs, cache_genes, excluded)
        cols = np.flatnonzero(hvg)
        X = harmony_trajectory(block, obs, cols, pca_mask=PASS1_HV[cols], n_comps=50, theta=6, seed=SEED,
                               dims=TRAJECTORY_DIMS)
        X = X[pt_rows] if arm == 'nephron' else X
        values = lognorm[pt_rows][:, marker_cols].toarray()
        score = marker_tip_score(values, species_pt, [0, 1, 2], [3, 4, 5])
        return X, score, None, cache_genes[cols[PASS1_HV[cols]]]
    block = lognorm[pt_rows]
    cols = cache_genes.get_indexer(list(anchors))
    if arm == 'cac':
        X = harmony_trajectory(block, neph_obs.iloc[pt_rows], cols, pca_mask=None, n_comps=CAC_PCS, theta=6,
                               seed=SEED, dims=TRAJECTORY_DIMS)
    else:  # 'cac_centered': no Harmony; each specimen centred on the mean of its S1 and S2 means
        dense = block[:, cols].toarray()
        for n in np.unique(specimen_pt):
            m = specimen_pt == n
            offset = np.mean([dense[m & (segment_pt == s)].mean(axis=0) for s in ('S1', 'S2')], axis=0)
            dense[m] -= offset
        obj = ad.AnnData(X=dense)
        sc.tl.pca(obj, n_comps=CAC_PCS, random_state=SEED, mask_var=None)
        X = np.asarray(obj.obsm['X_pca'][:, :TRAJECTORY_DIMS], float)
    return X, anchor_score(block, anchors), species_pt, pd.Index(anchors)


def run_arm(name, arm, counts, spec, exclude=(), anchors=None):
    def compute():
        X, score, groups, construction = embed(arm, counts, exclude, anchors)
        try:
            fit = fit_scfates_curve(X, score, nodes=SCFATES_NODES, seed=SEED, agree_groups=groups)
        except ValueError as error:
            # A refit whose curve cannot be rooted (branching, interior root, species disagreement) is
            # reported as failed, never repaired or retuned. Its DPT shares the root, so it fails too.
            nan = np.full(len(pt_rows), np.nan)
            return {'X': X, 'score': score, 'scfates': nan, 'root_cell': -1, 'dpt': nan,
                    'construction': np.asarray(construction, dtype=str), 'failed': str(error)}
        return {'X': X, 'score': score, 'scfates': fit['z'], 'root_cell': fit['root_cell'],
                'dpt': fit_dpt(X, fit['root_cell'], n_neighbors=PT_NEIGHBORS, seed=SEED),
                'construction': np.asarray(construction, dtype=str)}
    return cached_payload(f'arm_{name}', compute, root=cache_root,
                          params={'arm': arm, 'spec': spec, 'exclude': sorted(exclude),
                                  'anchors': sorted(anchors) if anchors is not None else None, 'seed': SEED},
                          inputs={'base_counts': BASE_DIGEST}, code=code)


reference = run_arm('scf13_full', 'nephron', COUNTS, 'unperturbed')
repro = {'scFates max |Δ|': float(np.abs(reference['scfates'] - SCF_SAVED).max()),
         'DPT max |Δ|': float(np.nanmax(np.abs(reference['dpt'] - DPT_SAVED))),
         'construction genes': len(reference['construction'])}
display(pd.Series(repro, name='reproduction of notebook 13'))
if repro['scFates max |Δ|'] > 1e-9 or repro['DPT max |Δ|'] > 1e-6 or repro['construction genes'] != 611:
    raise ValueError(f'The SCF13 arm does not reproduce notebook 13: {repro}')

### Perturbed inputs, shared by every arm

Equal depth uses the pooled PT 20th percentile of the input's measured-in-both library; the
count split is notebook 35's (seed 35). An arm built on equal-depth counts applies its own rule to
whatever it receives, including half-read inputs.

In [ ]:
def equal_depth(counts, seed):
    library = np.asarray(counts[:, measured_both].sum(axis=1)).ravel().astype(float)
    target = float(np.quantile(library[pt_rows], EQUAL_DEPTH_QUANTILE))
    return binomial_thin_to_depth(counts, library, target, np.random.default_rng(seed))


ED_352, ED_353 = equal_depth(COUNTS, 352), equal_depth(COUNTS, 353)
SPLIT_A, SPLIT_B = poisson_count_split(COUNTS, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))
ED_A, ED_B = equal_depth(SPLIT_A, 354), equal_depth(SPLIT_B, 355)


def hash_int(gene):
    return int(sha256(str(gene).encode()).hexdigest(), 16)


def inject(counts, genes, s0, seed):
    """Human-only exp(±log 2 (2 s0 − 1)) on the listed genes; signs alternate in hash order."""
    genes = sorted(genes, key=hash_int)
    signs = np.where(np.arange(len(genes)) % 2 == 0, 1.0, -1.0)
    f = INJECTION_LOG_AMPLITUDE * (2 * s0[human_pt] - 1)
    return scale_counts(counts, pt_rows[human_pt], genes_all.get_indexer(genes), f[:, None] * signs[None, :],
                        np.random.default_rng(seed)), genes, signs


NONANCHOR_INJECTED = sorted([g for g in CONSTRUCTION_GENES if not A_LOOSE.get(g, False)
                             and g not in EARLY_MARKERS + LATE_MARKERS and hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)
INJECTED_COUNTS, _, NONANCHOR_SIGNS = inject(COUNTS, NONANCHOR_INJECTED, SCF_SAVED, 351)
fold_genes = {k: tuple(cache_genes[FOLD.loc[cache_genes].to_numpy() == k]) for k in range(N_FOLDS)}
anchor_fold = pd.Series(sha256_folds(ANCHORS, N_FOLDS), index=ANCHORS)
anchor_half = pd.Series(sha256_folds(ANCHORS, 2), index=ANCHORS)

### Refits

Each arm is refitted with a third of its construction genes withheld (three folds; for CAC, a
third of the anchors), on equal-depth counts, on each count-split half, and, where planned, with
an injected species-by-position effect.

In [ ]:
FITS = {}
FITS['SCF13'] = {'full': reference,
                 **{f'fold{k}': run_arm(f'scf13_fold{k}', 'nephron', COUNTS, f'fold {k}', fold_genes[k]) for k in range(N_FOLDS)},
                 'ed': run_arm('scf13_ed', 'nephron', ED_352, 'equal depth 352'),
                 'half_A': run_arm('scf13_half_A', 'nephron', SPLIT_A, 'split A'),
                 'half_B': run_arm('scf13_half_B', 'nephron', SPLIT_B, 'split B')}
FITS['SCF13-ED'] = {'full': FITS['SCF13']['ed'],
                    **{f'fold{k}': run_arm(f'scf13ed_fold{k}', 'nephron', ED_352, f'ED 352 fold {k}', fold_genes[k]) for k in range(N_FOLDS)},
                    'ed': run_arm('scf13ed_second_draw', 'nephron', ED_353, 'equal depth 353'),
                    'half_A': run_arm('scf13ed_half_A', 'nephron', ED_A, 'ED of split A (354)'),
                    'half_B': run_arm('scf13ed_half_B', 'nephron', ED_B, 'ED of split B (355)')}
FITS['SCF-PT'] = {'full': run_arm('scfpt_full', 'pt', COUNTS, 'unperturbed'),
                  **{f'fold{k}': run_arm(f'scfpt_fold{k}', 'pt', COUNTS, f'fold {k}', fold_genes[k]) for k in range(N_FOLDS)},
                  'ed': run_arm('scfpt_ed', 'pt', ED_352, 'equal depth 352'),
                  'half_A': run_arm('scfpt_half_A', 'pt', SPLIT_A, 'split A'),
                  'half_B': run_arm('scfpt_half_B', 'pt', SPLIT_B, 'split B'),
                  'injection': run_arm('scfpt_injection', 'pt', INJECTED_COUNTS, 'non-anchor injection 351')}
cac_full = run_arm('cac_full', 'cac', COUNTS, 'unperturbed', anchors=ANCHORS)
ANCHOR_INJECTED = sorted([g for g in ANCHORS if hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)
ANCHOR_INJECTED_COUNTS, _, ANCHOR_SIGNS = inject(COUNTS, ANCHOR_INJECTED, cac_full['scfates'], 356)
FITS['CAC'] = {'full': cac_full,
               **{f'fold{k}': run_arm(f'cac_fold{k}', 'cac', COUNTS, f'anchor fold {k}',
                                      anchors=[g for g in ANCHORS if anchor_fold[g] != k]) for k in range(N_FOLDS)},
               **{f'half{h}': run_arm(f'cac_half{h}', 'cac', COUNTS, f'anchor half {h}',
                                      anchors=[g for g in ANCHORS if anchor_half[g] == h]) for h in range(2)},
               'ed': run_arm('cac_ed', 'cac', ED_352, 'equal depth 352', anchors=ANCHORS),
               'half_A': run_arm('cac_half_A', 'cac', SPLIT_A, 'split A', anchors=ANCHORS),
               'half_B': run_arm('cac_half_B', 'cac', SPLIT_B, 'split B', anchors=ANCHORS),
               'injection': run_arm('cac_injection', 'cac', INJECTED_COUNTS, 'non-anchor injection 351', anchors=ANCHORS),
               'anchor_injection': run_arm('cac_anchor_injection', 'cac', ANCHOR_INJECTED_COUNTS,
                                           'anchor injection 356', anchors=ANCHORS)}
SENSITIVITY = {'CAC-S49': run_arm('cac_strict', 'cac', COUNTS, 'strict anchors', anchors=ANCHORS_STRICT),
               'CAC-C': run_arm('cac_centered', 'cac_centered', COUNTS, 'centred, no Harmony', anchors=ANCHORS)}
# DPT13 shares SCF13's embeddings and roots.
FITS['DPT13'] = {key: {'scfates': fit['dpt'], 'X': fit['X']} for key, fit in FITS['SCF13'].items()}
ARMS = ['SCF13', 'SCF13-ED', 'DPT13', 'SCF-PT', 'CAC']
COORD = {arm: FITS[arm]['full']['scfates'] for arm in ARMS}
COORD.update({name: fit['scfates'] for name, fit in SENSITIVITY.items()})
pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
              'segment': segment_pt, **COORD}).to_csv(output / 'arm_coordinates.csv', index=False)
display(pd.DataFrame({arm: {f'{sp} {seg}': np.median(COORD[arm][(species_pt == sp) & (segment_pt == seg)])
                            for sp in SAMPLES for seg in ('S1', 'S2', 'S3')} for arm in COORD}).round(3)
        .rename_axis('median position'))

## 5 · Metrics for every arm

P1/P1w (fold-protected), gene-fold within-segment agreement, P5, registration gap, seed stability
and leave-one-specimen-out (scFates arms), held-out gene prediction (P4) and positive controls.
Registration and seed stability are descriptive and were added after notebook 35's outputs.

In [ ]:
EVAL_GENES = genes_all[(evaluation_mask('mouse', 'mouse_sn_male') | evaluation_mask('human', 'human_sn_halfB')
                        | evaluation_mask('mouse', 'mouse_micro') | D_OPPOSITE | D_FLAT).to_numpy()]
EVAL_X = EVAL_ORIGINAL[:, genes_all.get_indexer(EVAL_GENES)]
fold_of_eval = FOLD.loc[EVAL_GENES].to_numpy()
segment_group = np.where(human_pt, np.where(segment_pt == 'S3', 'S3', 'early'), segment_pt)
groups_within = np.char.add(np.char.add(specimen_pt.astype(str), '|'), segment_group.astype(str))
P1W_REFERENCES = {('mouse', 'S1'): 'mouse_micro_S2mS1', ('mouse', 'S2'): 'mouse_micro_S3mS1',
                  ('mouse', 'S3'): 'mouse_micro_S3mS2', ('human', 'early'): 'human_sn_halfB',
                  ('human', 'S3'): 'human_sn_halfB'}


def species_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], specimen_pt[keep])
    table.index = genes
    return pd.DataFrame({sp: table[names].mean(axis=1, skipna=False) for sp, names in SAMPLES.items()})


def within_r(expression, z, genes):
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], groups_within[keep])
    table.index = genes
    return table


def protected(arm, function):
    """Fold-k genes scored on the fold-k-withheld coordinate; CAC excludes every evaluation gene."""
    if arm == 'CAC':
        return function(EVAL_X, COORD['CAC'], EVAL_GENES)
    columns = function(EVAL_X[:, :1], COORD[arm], EVAL_GENES[:1]).columns
    parts = []
    for k in range(N_FOLDS):
        z, cols = FITS[arm][f'fold{k}']['scfates'], fold_of_eval == k
        parts.append(function(EVAL_X[:, cols], z, EVAL_GENES[cols]) if np.isfinite(z).any()
                     else pd.DataFrame(np.nan, index=EVAL_GENES[cols], columns=columns))  # failed refit
    return pd.concat(parts).loc[EVAL_GENES]


R, RW, metrics = {}, {}, {}
REFIT_FAILURES = {arm: [k for k, fit in FITS[arm].items() if not np.isfinite(fit['scfates']).any()] for arm in ARMS}
display(pd.Series({arm: ', '.join(v) or 'none' for arm, v in REFIT_FAILURES.items()}, name='failed refits'))
for arm in ARMS:
    if 'full' in REFIT_FAILURES[arm]:
        metrics[arm] = {'refit failures': len(REFIT_FAILURES[arm])}
        continue
    R[arm], RW[arm] = protected(arm, species_r), protected(arm, within_r)
    row = {'refit failures': len(REFIT_FAILURES[arm])}
    for sp, ref in (('mouse', 'mouse_sn_male'), ('mouse', 'mouse_micro'), ('human', 'human_sn_halfB')):
        mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
        row[f'P1 {sp} ({ref})'] = concordance(R[arm].loc[mask, sp], contrast.loc[EVAL_GENES[mask], ref])[0]
    for sp in SAMPLES:
        values = []
        for (s, seg), ref in P1W_REFERENCES.items():
            if s != sp:
                continue
            mask = evaluation_mask(sp, ref).loc[EVAL_GENES].to_numpy()
            values += [concordance(RW[arm].loc[mask, f'{n}|{seg}'], contrast.loc[EVAL_GENES[mask], ref])[0]
                       for n in SAMPLES[sp]]
        row[f'P1w {sp}'] = float(np.mean(values))
    pairs = [within_segment_agreement(FITS[arm][f'fold{a}']['scfates'], FITS[arm][f'fold{b}']['scfates'],
                                      specimen_pt, segment_pt) for a in range(N_FOLDS) for b in range(a + 1, N_FOLDS)]
    fold_agreement = pd.concat(pairs).groupby('segment').rho.median()
    row.update({f'gene-fold agreement {seg}': fold_agreement[seg] for seg in ('S1', 'S2', 'S3')})
    row['gene-fold agreement mean'] = float(fold_agreement.mean())
    if arm == 'SCF13-ED':  # two independent equal-depth draws
        p5 = within_segment_agreement(FITS[arm]['full']['scfates'], FITS[arm]['ed']['scfates'], specimen_pt, segment_pt)
    else:
        p5 = within_segment_agreement(FITS[arm]['ed']['scfates'], COORD[arm], specimen_pt, segment_pt)
    row['P5'] = float(p5.rho.median())
    row['P5 human S1'] = float(p5[p5.segment.eq('S1') & p5.specimen.isin(SAMPLES['human'])].rho.mean())
    gaps = {key: registration_gap(FITS[arm][key]['scfates'], species_pt, specimen_pt, segment_pt)[1]
            for key in ('fold0', 'fold1', 'fold2', 'half_A', 'half_B')}
    row.update({f'gap {k}': v for k, v in gaps.items()})
    row['registration gap'] = float(np.nanmax(list(gaps.values()))) if np.isfinite(list(gaps.values())).any() else np.nan
    row['registration refits failed'] = int(np.sum(~np.isfinite(list(gaps.values()))))
    row['saved-fit gap'] = registration_gap(COORD[arm], species_pt, specimen_pt, segment_pt)[1]
    ours = (R[arm].human - R[arm].mouse)
    ext_interaction = contrast['human_sn_halfB'] - contrast['mouse_sn_male']
    for label, mask in (('positive controls', D_OPPOSITE | D_FLAT),):
        idx = EVAL_GENES[mask.loc[EVAL_GENES].to_numpy()]
        ok = ours.loc[idx].notna()
        row['positive-control sign agreement'] = float(
            (np.sign(ours.loc[idx][ok]) == np.sign(ext_interaction.loc[idx][ok])).mean())
    metrics[arm] = row
METRICS = pd.DataFrame(metrics).T

# Seed stability and leave-one-specimen-out, scFates arms only (DPT is deterministic).
seed_rows, loso_rows = [], []
for arm in [a for a in ('SCF13', 'SCF13-ED', 'SCF-PT', 'CAC') if 'full' not in REFIT_FAILURES[a]]:
    full = FITS[arm]['full']
    groups = species_pt if arm == 'CAC' else None

    def seeds(full=full, groups=groups):
        out = {}
        for s in SEEDS:
            try:
                out[f'seed{s}'] = fit_scfates_curve(full['X'], full['score'], nodes=SCFATES_NODES, seed=s,
                                                    agree_groups=groups)['z']
            except ValueError:
                out[f'seed{s}'] = np.full(len(pt_rows), np.nan)  # failed seed, kept visible
        return out

    def loso(full=full, groups=groups):
        out = np.full(len(specimen_pt), np.nan)
        for n in np.unique(specimen_pt):
            try:
                out[specimen_pt == n] = loso_positions(full['X'], full['score'], specimen_pt, n, nodes=SCFATES_NODES,
                                                       seed=SEED, k=15, agree_groups=groups)
            except ValueError:
                pass  # failed held-out fit stays NaN
        return {'loso': out}
    key_inputs = {'X': full['X'], 'score': full['score']}
    by_seed = cached_payload(f'seeds_{arm}', seeds, root=cache_root, params={'seeds': SEEDS}, inputs=key_inputs, code=code)
    for s, z in by_seed.items():
        seed_rows.append(within_segment_agreement(z, full['scfates'], specimen_pt, segment_pt).assign(arm=arm, seed=s))
    held = cached_payload(f'loso_{arm}', loso, root=cache_root, params={'k': 15}, inputs=key_inputs, code=code)['loso']
    loso_rows.append(within_segment_agreement(held, full['scfates'], specimen_pt, segment_pt).assign(arm=arm))
SEED_STABILITY = pd.concat(seed_rows)
SEED_STABILITY['failed'] = SEED_STABILITY.rho.isna()
LOSO = pd.concat(loso_rows)
METRICS['seed stability'] = SEED_STABILITY.groupby('arm').rho.median()
METRICS['LOSO within-segment'] = LOSO.groupby('arm').rho.median()
SEED_STABILITY.to_csv(output / 'seed_stability.csv', index=False)
LOSO.to_csv(output / 'loso_within_segment.csv', index=False)

### P4 · Held-out gene prediction within species

For each species, fold and direction (one specimen trains, the other is predicted), fold-k genes
are predicted from a spline on the fold-k-withheld coordinate. The gain is relative to the
training specimen's segment means (privileged: it uses the query's labels). A quadratic
segment + coverage oracle (log library, log spots, log detected genes) is a second reference.

In [ ]:
covariates = np.column_stack([np.log(LIBRARY_PT), np.log(neph_obs.n_spots.to_numpy(float)[pt_rows]),
                              np.log(np.asarray((COUNTS[pt_rows] > 0).sum(axis=1)).ravel())])
anatomy_code = pd.Series(segment_pt).map({'S1': 0, 'S2': 1, 'S3': 2}).to_numpy()
p4_rows = []
for sp, names in SAMPLES.items():
    genes_sp = evaluation_mask(sp, 'mouse_sn_male' if sp == 'mouse' else 'human_sn_halfB').loc[EVAL_GENES].to_numpy()
    for k in range(N_FOLDS):
        cols = genes_sp & (fold_of_eval == k)
        for train, test in ((names[0], names[1]), (names[1], names[0])):
            tr, te = specimen_pt == train, specimen_pt == test
            ytr, yte = EVAL_X[tr][:, cols].toarray(), EVAL_X[te][:, cols].toarray()
            oracle = fit_segment_covariate_oracle(ytr, covariates[tr], anatomy_code[tr], specimen_pt[tr], degree=2)
            quad = np.mean((yte - predict_segment_covariate_oracle(oracle, covariates[te], anatomy_code[te])) ** 2, axis=0)
            for arm in ARMS:
                z = COORD['CAC'] if arm == 'CAC' else FITS[arm][f'fold{k}']['scfates']
                if not (np.isfinite(z[tr]).any() and np.isfinite(z[te]).any()):
                    p4_rows.append({'arm': arm, 'species': sp, 'fold': k, 'train': train,
                                    'gain vs segment oracle': np.nan, 'gain vs quadratic coverage oracle': np.nan})
                    continue
                out = heldout_prediction(ytr, z[tr], segment_pt[tr], yte, z[te], segment_pt[te], basis_df=BASIS_DF)
                var = np.maximum(out['train_var'], 1e-12)
                p4_rows.append({'arm': arm, 'species': sp, 'fold': k, 'train': train,
                                'gain vs segment oracle': 1 - np.sum(out['coordinate_mse'] / var) / np.sum(out['segment_mse'] / var),
                                'gain vs quadratic coverage oracle': 1 - np.sum(out['coordinate_mse'] / var) / np.sum(quad / var)})
P4 = pd.DataFrame(p4_rows)
P4.to_csv(output / 'p4_heldout_prediction.csv', index=False)
for sp in SAMPLES:
    METRICS[f'P4 gain {sp}'] = P4[P4.species.eq(sp)].groupby('arm')['gain vs segment oracle'].mean()
    METRICS[f'P4 gain vs quadratic {sp}'] = P4[P4.species.eq(sp)].groupby('arm')['gain vs quadratic coverage oracle'].mean()

### Absorption (P2) for the new arms and anchor misspecification for CAC (P2b)

In [ ]:
injected_eval, _ = pt_lognorm(INJECTED_COUNTS)
inj_cols = genes_all.get_indexer(NONANCHOR_INJECTED)


def interaction(expression, z, genes):
    r = species_r(expression, z, pd.Index(genes))
    return (r.human - r.mouse).to_numpy()


absorption = {}
for arm in [a for a in ('SCF-PT', 'CAC') if np.isfinite(FITS[a]['injection']['scfates']).any()
            and 'full' not in REFIT_FAILURES[a]]:
    original = interaction(EVAL_ORIGINAL[:, inj_cols], COORD[arm], NONANCHOR_INJECTED)
    frozen = interaction(injected_eval[:, inj_cols], COORD[arm], NONANCHOR_INJECTED)
    refit = interaction(injected_eval[:, inj_cols], FITS[arm]['injection']['scfates'], NONANCHOR_INJECTED)
    moved = within_segment_agreement(FITS[arm]['injection']['scfates'], COORD[arm], specimen_pt, segment_pt)
    absorption[arm] = {**absorption_ratio(original, frozen, refit, floor=ABSORPTION_FLOOR),
                       'human within-segment rho': float(moved[moved.specimen.isin(SAMPLES['human'])].rho.median())}
moved = within_segment_agreement(FITS['CAC']['anchor_injection']['scfates'], COORD['CAC'], specimen_pt, segment_pt)
P2B_HUMAN = float(moved[moved.specimen.isin(SAMPLES['human'])].rho.median())  # NaN if either fit failed
ABSORPTION = pd.DataFrame(absorption).T
ABSORPTION.to_csv(output / 'p2_absorption_new_arms.csv')
display(ABSORPTION.round(3))
print(f'P2b (10% of anchors made species-specific): human within-segment rho {P2B_HUMAN:.3f}')

### Anchor-half agreement and the attenuation index for CAC

Criterion 3 of the CAC rule needs agreement between coordinates built from the two anchor
halves. The attenuation index for CAC uses the coordinate from anchor half 0 and the strict
anchors in half 1, which that coordinate never saw.

In [ ]:
half_agreement = within_segment_agreement(FITS['CAC']['half0']['scfates'], FITS['CAC']['half1']['scfates'],
                                          specimen_pt, segment_pt)
CAC_HALF_AGREEMENT = float(half_agreement.rho.median())
print(f'CAC anchor-half within-segment agreement: median {CAC_HALF_AGREEMENT:.3f}')

## 6 · Notebook 12's model on each coordinate

Notebook 12's gene universe, matching strata and pathway family stay fixed; each coordinate gets
its own 1–99% support trimming. `T_spatial` is the species-by-smooth-position term; pathways are
called by the covariate-matched rank-AUC (strategy 8, q_normal ≤ 0.05). The specificity gate
repeats notebook 31's balanced relabeling, with the species shape as nuisance. Every coordinate is
run on the full universe and with the conserved anchors removed.

In [ ]:
from pseudospace.pathway_calibration import (balanced_partitions, contrast_designs, nested_partial_f,
                                             normal_matched_q)
from pseudospace.pathway_remodeling import (fit_nested_trajectories, gene_covariates, local_pathway_curves,
                                            matched_pathway_tests, matching_strata)
from pseudospace.pathways import build_pathway_membership
from pseudospace.stats_gam import gam_internal_knots
import pseudospace.pathway_remodeling as remodeling

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')


def support_rows(z):
    frame = pd.DataFrame({'z': z, 'specimen': specimen_pt})
    q = frame.groupby('specimen').z.quantile([.01, .99]).unstack()
    return np.flatnonzero(np.isfinite(z) & (z >= q[.01].max()) & (z <= q[.99].min()))


def common_grid(z, specimen, n=61):
    lo = max(z[specimen == s].min() for s in np.unique(specimen))
    hi = min(z[specimen == s].max() for s in np.unique(specimen))
    return np.linspace(lo, hi, n)


KEEP12 = support_rows(SCF_SAVED)
universe = gene_covariates(EVAL_ORIGINAL[KEEP12], SCF_SAVED[KEEP12], human_pt[KEEP12], specimen_pt[KEEP12], genes_all)
universe['eligible'] = measured_both & universe.detection.ge(.02).to_numpy()
UNIVERSE = pd.Index(universe.loc[universe.eligible, 'gene'])
STRATA = pd.Series(matching_strata(universe.set_index('gene').loc[UNIVERSE], bins=3), index=UNIVERSE)
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()),
    universe.loc[measured_both, 'gene'], ortholog_map=orthologs, library_name=lib, tested=list(UNIVERSE), min_genes=1)
    for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(UNIVERSE) - 1))]
GENE_SETS = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
saved_stats = pd.read_csv(upstream12 / 'gene_statistics.csv', index_col=0)
if list(saved_stats.index) != list(UNIVERSE):
    raise ValueError('Notebook 12 universe does not reproduce; rerun notebook 12.')
robustness = pd.read_csv(upstream33 / 'confident_pathways_with_robustness.csv')
ROBUST = robustness.loc[robustness.robustness.eq('robust'), 'pathway_id'].tolist()
CORE = robustness.loc[robustness.core.astype(bool), 'pathway_id'].tolist()
PARTITIONS = balanced_partitions(specimen_pt, species_pt)
pathway_code = digest([code, Path(remodeling.__file__)])
ANCHOR_SET = set(genes_all[A_LOOSE.to_numpy()])


def universe_variant(drop_anchors):
    genes = pd.Index([g for g in UNIVERSE if not (drop_anchors and g in ANCHOR_SET)])
    sets = {p: [g for g in m if g in set(genes)] for p, m in GENE_SETS.items()}
    sets = {p: m for p, m in sets.items() if 10 <= len(m) <= min(300, len(genes) - 1)}
    return genes, sets


def downstream(name, z, *, drop_anchors=False, expression=None):
    """Species and relabeled T_spatial, strategy-8 calls, NCDR and retention for one coordinate."""
    expression = EVAL_ORIGINAL if expression is None else expression
    genes, sets = universe_variant(drop_anchors)
    rows = support_rows(z)
    s, spec, hum = z[rows], specimen_pt[rows], human_pt[rows]
    Y = expression[rows][:, genes_all.get_indexer(genes)]
    knots = gam_internal_knots(s, basis_df=BASIS_DF)
    calls, stats = {}, {}
    for label, members in PARTITIONS.items():
        group = hum.astype(float) if label == 'species' else np.isin(spec, members).astype(float)
        designs, weights = contrast_designs(s, group, spec, knots,
                                            nuisance_shape=None if label == 'species' else hum.astype(float))
        t_spatial = nested_partial_f(Y, designs, weights, {'T_spatial': ('level', 'full')})['T_spatial']
        frame = pd.DataFrame({'T_spatial': t_spatial}, index=genes)
        tests = cached_frame(f'matched_{name}_{"noanchor" if drop_anchors else "full"}_{label.replace(":", "_").replace("+", "_")}',
                             lambda frame=frame, sets=sets: matched_pathway_tests(frame, sets, STRATA.loc[genes].to_numpy(),
                                                                                  n_null=N_NULL, seed=PATHWAY_SEED),
                             root=cache_root, params={'n_null': N_NULL, 'seed': PATHWAY_SEED},
                             inputs={'scores': frame, 'sets': sets}, code=pathway_code)
        calls[label] = normal_matched_q(tests).set_index('pathway_id').q_normal.le(ALPHA)
        stats[label] = frame.T_spatial
    relabeled = [calls[k].sum() for k in calls if k != 'species']
    species_calls = calls['species']
    robust_tested = [p for p in ROBUST if p in species_calls.index]
    core_tested = [p for p in CORE if p in species_calls.index]
    common = stats['species'].index.intersection(saved_stats.index)
    return {'coordinate': name, 'universe': 'anchors removed' if drop_anchors else 'full',
            'structures': len(rows), 'genes': len(genes), 'pathways tested': len(sets),
            'species calls': int(species_calls.sum()), 'relabeled calls (mean)': float(np.mean(relabeled)),
            'NCDR': float(np.mean(relabeled) / max(species_calls.sum(), 1)),
            'relabeled share of tested': float(np.mean(relabeled) / len(sets)),
            'specific': bool(np.mean(relabeled) / max(species_calls.sum(), 1) < .25 and np.mean(relabeled) / len(sets) < .05),
            'robust called': int(species_calls.loc[robust_tested].sum()), 'robust tested': len(robust_tested),
            'core called': int(species_calls.loc[core_tested].sum()), 'core tested': len(core_tested),
            'T_spatial Spearman vs notebook 12': spearmanr(stats['species'].loc[common], saved_stats.T_spatial.loc[common]).statistic,
            'median T_spatial': float(stats['species'].median())}, species_calls, stats['species']


def early_late_split(z):
    rows = support_rows(z)
    grid = common_grid(z[rows], specimen_pt[rows])
    fit = fit_nested_trajectories(EVAL_ORIGINAL[rows][:, genes_all.get_indexer(UNIVERSE)], z[rows], human_pt[rows],
                                  specimen_pt[rows], grid, basis_df=BASIS_DF)
    local = local_pathway_curves(fit['z'], list(UNIVERSE), {p: GENE_SETS[p] for p in ROBUST}, grid)
    peak = local.loc[local.groupby('pathway_id').divergence.idxmax()]
    human_high = peak.median_member_z.gt(0)
    return {'human-high': int(human_high.sum()), 'mouse-high': int((~human_high).sum()),
            'median peak human-high': float(peak.position[human_high].median()),
            'median peak mouse-high': float(peak.position[~human_high].median())}, fit, grid, rows


def attenuation(z, heldout):
    """Fraction of held-out conserved genes whose human amplitude is below the mouse one."""
    _, fit, grid, rows = early_late_split(z)
    seg, spec = segment_pt[rows], specimen_pt[rows]
    s_S1 = np.mean([np.median(z[rows][(spec == n) & (seg == 'S1')]) for n in np.unique(spec)])
    s_S3 = np.mean([np.median(z[rows][(spec == n) & (seg == 'S3')]) for n in np.unique(spec)])
    genes = [g for g in heldout if g in set(UNIVERSE)]
    idx = UNIVERSE.get_indexer(genes)
    sign = anchor_sign.loc[genes].to_numpy()
    amp = {sp: (interpolate_curves(fit[sp][idx], grid, [s_S3])[:, 0]
                - interpolate_curves(fit[sp][idx], grid, [s_S1])[:, 0]) * sign for sp in SAMPLES}
    pct = pd.Series(fit['T_spatial'], index=UNIVERSE).rank(pct=True).loc[genes]
    fraction = float(np.mean(amp['human'] < amp['mouse']))
    return {'held-out conserved genes': len(genes), 'fraction human < mouse': fraction,
            'median T_spatial percentile': float(pct.median()),
            'flag': bool(fraction >= .70 or pct.median() >= .75)}


attenuation_table = {}
for arm in ARMS:
    if 'full' in REFIT_FAILURES[arm] or (arm == 'CAC' and 'half0' in REFIT_FAILURES[arm]):
        continue
    construction = set(FITS[arm]['full'].get('construction', FITS['SCF13']['full']['construction']))
    if arm == 'CAC':
        heldout = [g for g in ANCHORS_STRICT if anchor_half[g] == 1]
        attenuation_table[arm] = attenuation(FITS['CAC']['half0']['scfates'], heldout)
    else:
        heldout = [g for g in genes_all[A_STRICT.to_numpy()] if g not in construction]
        attenuation_table[arm] = attenuation(COORD[arm], heldout)
ATTENUATION = pd.DataFrame(attenuation_table).T
ATTENUATION.to_csv(output / 'attenuation_by_arm.csv')
display(ATTENUATION)

## 7 · The CAC rule (C6), applied exactly as pre-registered

1. Non-inferior to SCF13: P1 and P1w ≥ SCF13 − 0.05 in each species; P4 gain ≥ SCF13 − 1 point in
   each species; P5 ≥ SCF13 − 0.05.
2. At least one advantage not favoured by construction: (a) SCF13 absorbs materially; (b) positive
   controls ≥ SCF13 + 10 points; (c) attenuation flag for SCF13 and not for CAC. Given notebook 35,
   none of these can be met (recorded in the plan before this run).
3. Stable: anchor-half agreement ≥ 0.70 and LOSO ≥ 0.80 (or ≥ SCF13 − 0.10).
4. P2b human within-segment ρ ≥ 0.80.
5. Downstream gate: CAC's `T_spatial` is specific.

In [ ]:
DOWNSTREAM_ROWS, CALLS, TSTATS = [], {}, {}
for name, z in (('SCF13', SCF_SAVED), ('CAC', COORD['CAC'])):
    if not np.isfinite(z).all():
        continue  # a failed CAC fit has no downstream screen and fails criterion 5
    for drop in (False, True):
        row, calls, stats = downstream(name, z, drop_anchors=drop)
        DOWNSTREAM_ROWS.append(row)
        CALLS[(name, drop)], TSTATS[(name, drop)] = calls, stats
# Guard, every run: on SCF13 the species T_spatial is notebook 12's.
if not np.allclose(TSTATS[('SCF13', False)].to_numpy(), saved_stats.T_spatial.to_numpy(), rtol=1e-6):
    raise ValueError('SCF13 T_spatial does not reproduce notebook 12.')
down = pd.DataFrame(DOWNSTREAM_ROWS).set_index(['coordinate', 'universe'])
m, b = METRICS.loc['CAC'], METRICS.loc['SCF13']
nb35 = results_root / 'pt_reconstruction_v2' / 'nb35'
nb35_displacement = pd.read_csv(nb35 / 'p2_displacement.csv')
P2_SCF13 = {'median_ratio': float(pd.read_csv(nb35 / 'p2_absorption.csv').set_index('coordinate').loc['SCF13', 'median_ratio']),
            'human within-segment rho': float(nb35_displacement[nb35_displacement.coordinate.eq('SCF13')
                                                                & nb35_displacement.species.eq('human')]['within-segment rho'].median())}
c6 = {
    '1 P1 mouse': m['P1 mouse (mouse_sn_male)'] >= b['P1 mouse (mouse_sn_male)'] - .05,
    '1 P1 human': m['P1 human (human_sn_halfB)'] >= b['P1 human (human_sn_halfB)'] - .05,
    '1 P1w mouse': m['P1w mouse'] >= b['P1w mouse'] - .05,
    '1 P1w human': m['P1w human'] >= b['P1w human'] - .05,
    '1 P4 mouse': m['P4 gain mouse'] >= b['P4 gain mouse'] - .01,
    '1 P4 human': m['P4 gain human'] >= b['P4 gain human'] - .01,
    '1 P5': m['P5'] >= b['P5'] - .05,
    '2a SCF13 absorbs': P2_SCF13['median_ratio'] < .85 or P2_SCF13['human within-segment rho'] < .90,
    '2b positive controls +10 points': m['positive-control sign agreement'] >= b['positive-control sign agreement'] + .10,
    '2c attenuation flag SCF13 only': bool('CAC' in ATTENUATION.index and ATTENUATION.loc['SCF13', 'flag']
                                           and not ATTENUATION.loc['CAC', 'flag']),
    '3 anchor-half agreement ≥ 0.70': CAC_HALF_AGREEMENT >= .70,
    '3 LOSO': m['LOSO within-segment'] >= .80 or m['LOSO within-segment'] >= b['LOSO within-segment'] - .10,
    '4 P2b ≥ 0.80': P2B_HUMAN >= .80,
    '5 downstream specific': bool(('CAC', 'full') in down.index and down.loc[('CAC', 'full'), 'specific']),
}
criterion = {'1': all(v for k, v in c6.items() if k.startswith('1')),
             '2': any(v for k, v in c6.items() if k.startswith('2')),
             '3': all(v for k, v in c6.items() if k.startswith('3')),
             '4': c6['4 P2b ≥ 0.80'], '5': c6['5 downstream specific']}
CAC_PASSES = all(criterion.values())
C6 = pd.Series({**c6, **{f'criterion {k}': v for k, v in criterion.items()}, 'CAC adopted under C6': CAC_PASSES})
C6.to_csv(output / 'c6_decision.csv')
display(C6.to_frame('passes'))

## 8 · Selecting the paper coordinate (rule fixed before this run)

Candidates: SCF13-ED, DPT13, and CAC only if it passed C6. A candidate qualifies if P5 ≥ 0.80,
P1 ≥ SCF13 − 0.05 in each species, and its registration gap is ≤ 0.15. The qualifying candidate
with the highest mean gene-fold within-segment agreement wins; within 0.02, the simpler and
stabler one is preferred (DPT over scFates). If none qualifies, SCF13 stays.

In [ ]:
candidates = ['SCF13-ED', 'DPT13'] + (['CAC'] if CAC_PASSES else [])
PREFERENCE = {'DPT13': 0, 'SCF13-ED': 1, 'CAC': 2}
selection = pd.DataFrame({arm: {
    'P5': METRICS.loc[arm, 'P5'],
    'P1 mouse': METRICS.loc[arm, 'P1 mouse (mouse_sn_male)'], 'P1 human': METRICS.loc[arm, 'P1 human (human_sn_halfB)'],
    'registration gap': METRICS.loc[arm, 'registration gap'],
    'gene-fold agreement mean': METRICS.loc[arm, 'gene-fold agreement mean']} for arm in candidates + ['SCF13']}).T
selection['(i) P5 ≥ 0.80'] = selection.P5 >= .80
selection['(ii) P1 ≥ SCF13 − 0.05'] = ((selection['P1 mouse'] >= b['P1 mouse (mouse_sn_male)'] - .05)
                                      & (selection['P1 human'] >= b['P1 human (human_sn_halfB)'] - .05))
selection['(iii) gap ≤ 0.15'] = selection['registration gap'] <= .15
selection['qualifies'] = selection[['(i) P5 ≥ 0.80', '(ii) P1 ≥ SCF13 − 0.05', '(iii) gap ≤ 0.15']].all(axis=1)
selection.loc['SCF13', 'qualifies'] = False  # SCF13 is the fallback, not a candidate
qualified = selection[selection.qualifies]
if qualified.empty:
    SELECTED = 'SCF13'
else:
    best = qualified['gene-fold agreement mean'].max()
    close = qualified[qualified['gene-fold agreement mean'] >= best - .02]
    SELECTED = min(close.index, key=lambda a: PREFERENCE[a])
selection.to_csv(output / 'selection_rule.csv')
display(selection.round(3))
print('Selected paper coordinate:', SELECTED)

SELECTED_Z = COORD[SELECTED] if SELECTED != 'SCF13' else SCF_SAVED
METHODS = {'SCF13': 'notebook 13 nonbranching scFates on pass-2 Harmony (611 PCA genes)',
           'SCF13-ED': 'notebook 13 pipeline on counts thinned to equal depth (pooled PT 20th percentile, seed 352)',
           'DPT13': 'scanpy DPT on notebook 13 pass-2 Harmony (first 5 dimensions), rooted at notebook 13 scFates root',
           'CAC': 'conserved-anchor scFates (external S3-versus-early anchors only)'}
selected_table = pd.DataFrame({'structure_id': saved_pt.index, 'specimen': specimen_pt, 'species': species_pt,
                               'segment_class': saved_pt.segment_class.astype(str).to_numpy(),
                               'position': SELECTED_Z, 'coordinate_method': SELECTED,
                               'method_description': METHODS[SELECTED]})
if not np.isfinite(selected_table.position).all():
    raise ValueError('The selected coordinate must be finite for every PT structure.')
SELECTED_PATH = results_root / 'pt_reconstruction_v2' / 'selected_pt_coordinate.csv'
selected_table.to_csv(SELECTED_PATH, index=False)
print('Saved', SELECTED_PATH, len(selected_table), 'structures')

## 9 · Downstream effect of the selected coordinate

Same model, calls and gate as notebook 31's strategy 8, on the selected coordinate and on CAC
(robustness), each with the full universe and with anchors removed. SCF13 is the reference. The
early/late split of the 66 robust pathways is recomputed on each coordinate.

In [ ]:
for name, z in ((SELECTED, SELECTED_Z),):
    if name == 'SCF13':
        continue
    for drop in (False, True):
        row, calls, stats = downstream(name, z, drop_anchors=drop)
        DOWNSTREAM_ROWS.append(row)
        CALLS[(name, drop)], TSTATS[(name, drop)] = calls, stats
DOWNSTREAM = pd.DataFrame(DOWNSTREAM_ROWS)
DOWNSTREAM.to_csv(output / 'downstream_summary.csv', index=False)
display(DOWNSTREAM.round(3))
splits = {name: early_late_split(z)[0] for name, z in (('SCF13', SCF_SAVED), ('CAC', COORD['CAC']), (SELECTED, SELECTED_Z))
          if np.isfinite(z).all()}
SPLITS = pd.DataFrame(splits).T
SPLITS.to_csv(output / 'early_late_split_by_coordinate.csv')
display(SPLITS)
overlap = {}
for name in sorted({SELECTED, 'CAC'}):
    for drop in (False, True):
        if (name, drop) not in CALLS:
            continue
        a, ref = CALLS[(name, drop)], CALLS[('SCF13', drop)]
        both = a.index.intersection(ref.index)
        overlap[f'{name} ({"anchors removed" if drop else "full"})'] = {
            'calls': int(a.sum()), 'SCF13 calls': int(ref.loc[both].sum()),
            'shared calls': int((a.loc[both] & ref.loc[both]).sum())}
display(pd.DataFrame(overlap).T)
pd.concat({f'{k[0]}|{"noanchor" if k[1] else "full"}': v for k, v in CALLS.items()}, axis=1).to_csv(
    output / 'pathway_calls_by_coordinate.csv')

### External species-by-position concordance on each coordinate

The coordinate-based interaction r_human − r_mouse is compared with the external interaction
(human snRNA half B minus male mouse snRNA) on non-anchor genes detected in both species, for all
such genes and for each coordinate's top 10% by `T_spatial`.

In [ ]:
external_interaction = contrast['human_sn_halfB'] - contrast['mouse_sn_male']
shared = control_eligible & level['human_sn_halfB'].ge(1) & external_interaction.notna()
shared_genes = genes_all[shared.to_numpy()]
shared_x = EVAL_ORIGINAL[:, genes_all.get_indexer(shared_genes)]
concordance_rows = []
for name, z in (('SCF13', SCF_SAVED), ('CAC', COORD['CAC']), (SELECTED, SELECTED_Z)):
    if (name, False) not in TSTATS:
        continue
    r = species_r(shared_x, z, shared_genes)
    ours = r.human - r.mouse
    t = TSTATS[(name, False)]
    top = t[t >= t.quantile(.9)].index
    for subset, idx in (('all shared genes', shared_genes), ('top 10% T_spatial', shared_genes.intersection(top))):
        rho, n = concordance(ours.loc[idx], external_interaction.loc[idx])
        strong = idx[np.abs(external_interaction.loc[idx]) >= 1]
        agree = float((np.sign(ours.loc[strong]) == np.sign(external_interaction.loc[strong])).mean())
        concordance_rows.append({'coordinate': name, 'genes': subset, 'n': n, 'Spearman': rho,
                                 'sign agreement, |external| ≥ 1': agree, 'n strong': len(strong)})
EXTERNAL = pd.DataFrame(concordance_rows).drop_duplicates(['coordinate', 'genes'])
EXTERNAL.to_csv(output / 'external_species_by_position.csv', index=False)
display(EXTERNAL.round(3))

### Count split on the selected coordinate

As in notebook 35: (a) coordinate refitted on half A and tested on half B; (b) the full-data
coordinate tested on half B; (c) refitted and tested on half B.

In [ ]:
eval_b, _ = pt_lognorm(SPLIT_B)
split_rows = []
if SELECTED != 'SCF13':
    for arm_label, z in (('a · from A (independent)', FITS[SELECTED]['half_A']['scfates']),
                         ('b · full-data coordinate', SELECTED_Z),
                         ('c · from B (circular)', FITS[SELECTED]['half_B']['scfates'])):
        row, calls, stats = downstream(f'{SELECTED}_split_{arm_label[0]}', z, expression=eval_b)
        split_rows.append({'arm': arm_label, **{k: row[k] for k in ('structures', 'species calls', 'robust called',
                                                                      'core called', 'median T_spatial', 'NCDR')}})
COUNT_SPLIT = pd.DataFrame(split_rows)
COUNT_SPLIT.to_csv(output / 'count_split_selected.csv', index=False)
display(COUNT_SPLIT)

## 10 · Summary

In [ ]:
METRICS.to_csv(output / 'arm_metrics.csv')
display(METRICS.T.round(3))
summary = {
    'logic_version': NOTEBOOK_LOGIC_VERSION, 'reproduction': repro,
    'anchors used': len(ANCHORS), 'metrics': json.loads(METRICS.to_json(orient='index')),
    'sensitivity arms within-segment agreement with CAC': {
        name: float(within_segment_agreement(z, COORD['CAC'], specimen_pt, segment_pt).rho.median())
        for name, z in ((n, SENSITIVITY[n]['scfates']) for n in SENSITIVITY)},
    'absorption': json.loads(ABSORPTION.to_json(orient='index')), 'P2b human rho': P2B_HUMAN,
    'CAC anchor-half agreement': CAC_HALF_AGREEMENT, 'attenuation': json.loads(ATTENUATION.to_json(orient='index')),
    'C6': {k: bool(v) for k, v in C6.items()}, 'selection': json.loads(selection.to_json(orient='index')),
    'selected': SELECTED, 'selected_path': str(SELECTED_PATH.relative_to(results_root)),
    'downstream': json.loads(DOWNSTREAM.to_json(orient='records')),
    'early_late_split': json.loads(SPLITS.to_json(orient='index')),
    'external': json.loads(EXTERNAL.to_json(orient='records')),
    'count_split_selected': json.loads(COUNT_SPLIT.to_json(orient='records')),
}
(output / 'summary.json').write_text(json.dumps(summary, indent=1, default=float))
print(json.dumps({k: summary[k] for k in ('C6', 'selected', 'selected_path')}, indent=1))